## 1. 설치 및 기본 import
* 필요한 라이브러리 준비
* random seed 고정
* GPU/CPU device 설정

In [1]:
# !pip install -q transformers datasets peft accelerate bitsandbytes evaluate sentencepiece scikit-learn
# !pip install -q rouge-score

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from datasets import Dataset as HFDataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## 2. 경로 설정 및 데이터 로드
* SFT / RM / PPO 파일 로드
* 데이터 구조 확인
* 이후 전처리 전 사전 점검

In [2]:
BASE_DIR = "/home/jovyan/work/project/first-repository/NLP/NLP04/KoChatGPT/data_kochatgpt"
# 경로 다르면 여기만 수정
SFT_PATH = os.path.join(BASE_DIR, "kochatgpt_1_SFT.jsonl")
RM_PATH  = os.path.join(BASE_DIR, "kochatgpt_2_RM.jsonl")
PPO_PATH = os.path.join(BASE_DIR, "kochatgpt_3_PPO.jsonl")

with open(SFT_PATH, "r", encoding="utf-8-sig") as f:
    sft_data = json.load(f)

with open(RM_PATH, "r", encoding="utf-8-sig") as f:
    rm_data = json.load(f)

with open(PPO_PATH, "r", encoding="utf-8-sig") as f:
    ppo_data = json.load(f)

print("SFT:", len(sft_data))
print("RM :", len(rm_data))
print("PPO:", len(ppo_data))

print(sft_data[0].keys())
print(rm_data[0].keys())
print(ppo_data[0].keys())

SFT: 12000
RM : 10220
PPO: 12000
dict_keys(['prompt', 'completion', 'tokens'])
dict_keys(['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking'])
dict_keys(['prompt'])


## 3. 데이터 EDA + 정제 함수
좋은 모델을 만들기 전에, 먼저 학습 데이터 품질을 개선한 단계
* 길이 분포 확인
* normalize_text로 공백/깨진 문자/반복 표현 정리
* bad sample 제거

In [3]:
def text_stats(texts):
    lengths = [len(t) for t in texts if isinstance(t, str)]
    return {
        "count": len(lengths),
        "mean": np.mean(lengths),
        "median": np.median(lengths),
        "min": np.min(lengths),
        "max": np.max(lengths),
        "p95": np.percentile(lengths, 95)
    }

def normalize_text(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = x.strip()
    x = re.sub(r"\s+", " ", x)                       # 공백 정리
    x = re.sub(r"[�]+", "", x)                      # 깨진 문자 제거
    x = re.sub(r"(.)\1{5,}", r"\1\1", x)            # 과도한 반복 축소
    return x

def is_bad_sample(prompt: str, completion: str) -> bool:
    # 너무 짧거나 비정상 패턴 제거
    if len(prompt.strip()) < 2 or len(completion.strip()) < 5:
        return True
    bad_patterns = [
        r"Allow me to answer your question",
        r"The diameter of the Metallic domain",
        r"^[A-Za-z0-9_\-\.,;: ]{25,}$",             # 영어/기호만 과도한 경우
    ]
    for p in bad_patterns:
        if re.search(p, completion):
            return True
    return False

# SFT 정제
clean_sft = []
for row in sft_data:
    p = normalize_text(row["prompt"])
    c = normalize_text(row["completion"])
    if not is_bad_sample(p, c):
        clean_sft.append({"prompt": p, "completion": c})

print("원본 SFT:", len(sft_data))
print("정제 SFT:", len(clean_sft))

# RM 정제
clean_rm = []
for row in rm_data:
    p = normalize_text(row["prompt"])
    cands = [normalize_text(row[f"completion_{i}"]) for i in range(3)]
    ranking = row["ranking"]

    # 3개 후보 모두 유효해야 사용
    if any(len(c) < 3 for c in cands):
        continue
    clean_rm.append({
        "prompt": p,
        "completion_0": cands[0],
        "completion_1": cands[1],
        "completion_2": cands[2],
        "ranking": ranking
    })

print("원본 RM:", len(rm_data))
print("정제 RM:", len(clean_rm))

# 간단 EDA
sft_prompt_stats = text_stats([x["prompt"] for x in clean_sft])
sft_completion_stats = text_stats([x["completion"] for x in clean_sft])

print("SFT prompt stats:", sft_prompt_stats)
print("SFT completion stats:", sft_completion_stats)

원본 SFT: 12000
정제 SFT: 11985
원본 RM: 10220
정제 RM: 10067
SFT prompt stats: {'count': 11985, 'mean': np.float64(22.183562786816854), 'median': np.float64(19.0), 'min': np.int64(2), 'max': np.int64(295), 'p95': np.float64(46.0)}
SFT completion stats: {'count': 11985, 'mean': np.float64(144.15027117229872), 'median': np.float64(118.0), 'min': np.int64(5), 'max': np.int64(1553), 'p95': np.float64(382.0)}


## 4. 베이스라인 모델 로드
* skt/kogpt2-base-v2 로드
* pad/eos token 세팅
* baseline 비교 기준 확보

In [4]:
BASE_MODEL_NAME = "skt/kogpt2-base-v2"

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    padding_side="right",
)

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.eos_token_id = tokenizer.eos_token_id
base_model = base_model.to(device)

print("tokenizer pad:", tokenizer.pad_token_id)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer pad: 3


## 5. 프롬프트 템플릿 통일
* <usr> / <sys> 템플릿 정의
* 학습-추론 형식 통일
* 응답 일관성 확보

In [5]:
def build_prompt(user_text: str) -> str:
    return f"<usr> {user_text}\n<sys>"

def build_full_text(user_text: str, assistant_text: str) -> str:
    return f"<usr> {user_text}\n<sys> {assistant_text}{tokenizer.eos_token}"

## 6. Baseline 생성 함수
* baseline 응답 생성
* 디코딩 파라미터 포함
* 이후 SFT와 비교할 기준 결과 생성

In [6]:
@torch.no_grad()
def generate_response(
    model,
    prompt,
    max_new_tokens=64,
    do_sample=True,
    top_p=0.9,
    top_k=50,
    temperature=0.8,
    num_beams=1,
):
    model.eval()
    text = build_prompt(prompt)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample if num_beams == 1 else False,
        top_p=top_p,
        top_k=top_k,
        temperature=temperature,
        num_beams=num_beams,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # <sys> 뒤만 추출
    if "<sys>" in decoded:
        answer = decoded.split("<sys>", 1)[1]
    else:
        answer = decoded
    answer = answer.replace(tokenizer.eos_token, "").strip()
    return answer

test_prompts = [
    "불고기용 고기 한우에요?",
    "쓰던 앱이 유료로 전환됐어",
    "여친이랑 다툼",
    "잠이 안 와서 너무 힘들어",
    "면접 전에 너무 긴장돼",
]

baseline_results = []
for p in test_prompts:
    out = generate_response(base_model, p)
    baseline_results.append({"prompt": p, "baseline": out})

pd.DataFrame(baseline_results)

,prompt,baseline
0,불고기용 고기 한우에요?,}▁�r�t<unk><unk>▁(▁�r��e�▁)▁이다.\n이▁역은▁러시아▁제국▁시...
1,쓰던 앱이 유료로 전환됐어,"?""("")""이다.\n이런▁이유로▁중국▁정부는▁베이징▁올림픽을▁중화민국의▁올림픽으로▁..."
2,여친이랑 다툼,▁<unk><unk>▁<unk>ại<unk>▁�n�\n2017-10-13▁|▁14...
3,잠이 안 와서 너무 힘들어,")▁또는▁녜띠(<unk>�n�t),▁녜타오(<unk>ta<unk>)로▁표기했다.\n..."
4,면접 전에 너무 긴장돼,<unk>ん<unk><unk>ん\n[[[]]]\n[[1]]\n[▁[]]\n*[]]\...


## 7. SFT 학습용 데이터셋 만들기
모델이 답변 생성 부분만 배우도록 만든 셀
* full text와 prompt 분리
* prompt 부분 label = -100 처리
* completion만 학습

In [7]:
MAX_LEN = 256

class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        full_text = build_full_text(row["prompt"], row["completion"])
        prompt_text = build_prompt(row["prompt"])

        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        prompt_enc = self.tokenizer(
            prompt_text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = full_enc["input_ids"][0]
        attention_mask = full_enc["attention_mask"][0]
        labels = input_ids.clone()

        prompt_len = int(prompt_enc["attention_mask"][0].sum())
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

train_sft, valid_sft = train_test_split(clean_sft, test_size=0.1, random_state=SEED)

train_dataset = SFTDataset(train_sft, tokenizer, MAX_LEN)
valid_dataset = SFTDataset(valid_sft, tokenizer, MAX_LEN)

len(train_dataset), len(valid_dataset)

(10786, 1199)

## 8. LoRA 적용한 SFT 모델 학습
* LoRA 적용
* 적은 자원으로 효율적 튜닝
* Trainer 기반 SFT 학습

In [8]:
sft_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
sft_model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],  # GPT2 계열
)

sft_model = get_peft_model(sft_model, lora_config)
sft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./results_sft",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    eval_strategy="steps",   # 여기 수정
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=sft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
trainer.save_model("./kogpt2_sft_lora")
tokenizer.save_pretrained("./kogpt2_sft_lora")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/opt/conda/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 811,008 || all params: 165,296,640 || trainable%: 0.4906


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,1.905106,1.845194
400,1.835113,1.756605
600,1.748976,1.706988
800,1.722004,1.678634
1000,1.710769,1.658814
1200,1.685808,1.649857


('./kogpt2_sft_lora/tokenizer_config.json', './kogpt2_sft_lora/tokenizer.json')

## 9. SFT 결과 확인
* baseline vs SFT 직접 비교
* 정성 평가 표 생성

In [9]:
sft_results = []
for p in test_prompts:
    out = generate_response(sft_model, p)
    sft_results.append({"prompt": p, "sft": out})

compare_df = pd.DataFrame(baseline_results).merge(pd.DataFrame(sft_results), on="prompt")
compare_df

,prompt,baseline,sft
0,불고기용 고기 한우에요?,}▁�r�t<unk><unk>▁(▁�r��e�▁)▁이다.\n이▁역은▁러시아▁제국▁시...,'��AI����������챰������������.
1,쓰던 앱이 유료로 전환됐어,"?""("")""이다.\n이런▁이유로▁중국▁정부는▁베이징▁올림픽을▁중화민국의▁올림픽으로▁...","'��AI���.��������������,��������������."
2,여친이랑 다툼,▁<unk><unk>▁<unk>ại<unk>▁�n�\n2017-10-13▁|▁14...,"'��AI��������,�거������������������.��������������"
3,잠이 안 와서 너무 힘들어,")▁또는▁녜띠(<unk>�n�t),▁녜타오(<unk>ta<unk>)로▁표기했다.\n...","'���AI��,�������������."
4,면접 전에 너무 긴장돼,<unk>ん<unk><unk>ん\n[[[]]]\n[[1]]\n[▁[]]\n*[]]\...,"'�AI���������,����������������."


## 10. RM용 pairwise 데이터 생성
* ranking → chosen / rejected pair 변환
* RM 학습용 데이터 준비

In [10]:
def ranking_to_pairs(row):
    comps = [row["completion_0"], row["completion_1"], row["completion_2"]]
    ranking = row["ranking"]  # 예: [2,1,0] 이면 2번이 최고
    ordered = ranking[:]      # best -> worst 인덱스 순서라고 가정

    pairs = []
    for i in range(len(ordered)):
        for j in range(i+1, len(ordered)):
            better_idx = ordered[i]
            worse_idx = ordered[j]
            pairs.append({
                "prompt": row["prompt"],
                "chosen": comps[better_idx],
                "rejected": comps[worse_idx],
            })
    return pairs

rm_pairs = []
for row in clean_rm:
    rm_pairs.extend(ranking_to_pairs(row))

print("RM pair 개수:", len(rm_pairs))
print(rm_pairs[0])

RM pair 개수: 30201
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'chosen': '라이언에게 말했다.', 'rejected': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.'}


## 11. Reward Model 정의
* backbone + value head
* 응답마다 보상 점수 예측
* pairwise preference 학습 기반

In [11]:
class RewardDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=256):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row = self.pairs[idx]

        chosen_text = build_full_text(row["prompt"], row["chosen"])
        rejected_text = build_full_text(row["prompt"], row["rejected"])

        chosen = self.tokenizer(
            chosen_text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        rejected = self.tokenizer(
            rejected_text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )

        return {
            "chosen_input_ids": chosen["input_ids"][0],
            "chosen_attention_mask": chosen["attention_mask"][0],
            "rejected_input_ids": rejected["input_ids"][0],
            "rejected_attention_mask": rejected["attention_mask"][0],
        }

class RewardModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.value_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # [B, L, H]
        last_idx = attention_mask.sum(dim=1) - 1
        pooled = hidden[torch.arange(hidden.size(0)), last_idx]  # 마지막 토큰
        reward = self.value_head(pooled).squeeze(-1)
        return reward

def rm_collate_fn(batch):
    out = {}
    for k in batch[0].keys():
        out[k] = torch.stack([x[k] for x in batch])
    return out

train_rm, valid_rm = train_test_split(rm_pairs, test_size=0.1, random_state=SEED)

train_rm_ds = RewardDataset(train_rm, tokenizer, MAX_LEN)
valid_rm_ds = RewardDataset(valid_rm, tokenizer, MAX_LEN)

train_rm_loader = DataLoader(train_rm_ds, batch_size=8, shuffle=True, collate_fn=rm_collate_fn)
valid_rm_loader = DataLoader(valid_rm_ds, batch_size=8, shuffle=False, collate_fn=rm_collate_fn)

## 12. RM 학습
* pairwise ranking loss
* chosen > rejected 되도록 학습
* validation accuracy로 성능 확인

In [12]:
reward_model = RewardModel(BASE_MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(reward_model.parameters(), lr=2e-5)

def reward_loss(chosen_reward, rejected_reward):
    return -torch.log(torch.sigmoid(chosen_reward - rejected_reward) + 1e-8).mean()

def evaluate_rm(model, dataloader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            chosen_ids = batch["chosen_input_ids"].to(device)
            chosen_mask = batch["chosen_attention_mask"].to(device)
            rejected_ids = batch["rejected_input_ids"].to(device)
            rejected_mask = batch["rejected_attention_mask"].to(device)

            chosen_reward = model(chosen_ids, chosen_mask)
            rejected_reward = model(rejected_ids, rejected_mask)

            loss = reward_loss(chosen_reward, rejected_reward)
            total_loss += loss.item()

            correct += (chosen_reward > rejected_reward).sum().item()
            total += chosen_reward.size(0)

    return total_loss / len(dataloader), correct / total

best_acc = 0.0
for epoch in range(3):
    reward_model.train()
    total_train_loss = 0

    for batch in train_rm_loader:
        chosen_ids = batch["chosen_input_ids"].to(device)
        chosen_mask = batch["chosen_attention_mask"].to(device)
        rejected_ids = batch["rejected_input_ids"].to(device)
        rejected_mask = batch["rejected_attention_mask"].to(device)

        chosen_reward = reward_model(chosen_ids, chosen_mask)
        rejected_reward = reward_model(rejected_ids, rejected_mask)

        loss = reward_loss(chosen_reward, rejected_reward)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    val_loss, val_acc = evaluate_rm(ㄴreward_model, valid_rm_loader)
    print(f"[Epoch {epoch+1}] train_loss={total_train_loss/len(train_rm_loader):.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(reward_model.state_dict(), "./best_reward_model.pt")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'ᄂreward_model' is not defined

## 13. SFT 후보 4개 생성 후 RM으로 재랭킹
* 후보 여러 개 생성
* RM이 점수 부여
* 최고 점수 응답 선택

In [13]:
reward_model.load_state_dict(torch.load("./best_reward_model.pt", map_location=device))
reward_model.eval()

@torch.no_grad()
def score_with_rm(prompt, answer):
    text = build_full_text(prompt, answer)
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding="max_length").to(device)
    reward = reward_model(enc["input_ids"], enc["attention_mask"])
    return reward.item()

@torch.no_grad()
def generate_candidates(model, prompt, n=4, max_new_tokens=64):
    model.eval()
    text = build_prompt(prompt)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        temperature=0.8,
        num_return_sequences=n,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    candidates = []
    for out in outputs:
        decoded = tokenizer.decode(out, skip_special_tokens=False)
        ans = decoded.split("<sys>", 1)[1] if "<sys>" in decoded else decoded
        ans = ans.replace(tokenizer.eos_token, "").strip()
        candidates.append(ans)
    return candidates

def sft_rm_answer(model, prompt, n_candidates=4):
    candidates = generate_candidates(model, prompt, n=n_candidates)
    scored = [(cand, score_with_rm(prompt, cand)) for cand in candidates]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    best_answer, best_score = scored[0]
    return best_answer, scored

## 14. Baseline / SFT / SFT+RM 비교 표 만들기
* 3단계 결과 비교
* baseline → SFT → SFT+RM 흐름 확인

In [14]:
rows = []
for p in test_prompts:
    baseline_out = generate_response(base_model, p)
    sft_out = generate_response(sft_model, p)
    rm_best, rm_scored = sft_rm_answer(sft_model, p, n_candidates=4)

    rows.append({
        "prompt": p,
        "baseline": baseline_out,
        "sft": sft_out,
        "sft_rm_best": rm_best,
        "rm_scores": rm_scored
    })

result_df = pd.DataFrame(rows)
result_df

,prompt,baseline,sft,sft_rm_best,rm_scores
0,불고기용 고기 한우에요?,示<unk>[sː<unk>][s�ː<unk>]\n<unk>���z�z���<un...,'��AI�������������.\n\n1.���������������������...,"'��������,�������������������.","[('��������,�������������������., -5.332963943..."
1,쓰던 앱이 유료로 전환됐어,)▁중국▁인민해방군▁군인들이▁전방을▁오가면서▁중국▁인민군이▁진격하기▁시작했다.\n1...,"'����,����������������������.","'�AI�����,������.<pad><pad><pad><pad><pad><pad...","[('�AI�����,������.<pad><pad><pad><pad><pad><p..."
2,여친이랑 다툼,(<unk>����)라▁했다.\n중국▁정부는▁중국▁내▁소수민족을▁위한▁각종▁지원을▁...,"'���AI����.�������������������,���������������...","'��AI���,����������������.���������������������","[('��AI���,����������������.������������������..."
3,잠이 안 와서 너무 힘들어,▁��n�ti�li�iǎn�n��e▁re�r�ric�ta�r�▁à▁ph��te�\...,'���AI���.,"'��AI��,����������������.�����������������������!","[('��AI��,����������������.�������������������..."
4,면접 전에 너무 긴장돼,▁?�n��r�t��<unk>�i�n�<unk>��l�����u�n�te����...,"'�������,�챰�거��������.",'��AI���.\n\n2!\n3!�������������������������거�...,[('��AI���.\n\n2!\n3!�������������������������...


## 15. 정량평가 코드
* BLEU / ROUGE 계산
* 평균 길이, 반복률 측정
* 정량 비교 근거 확보

In [15]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# validation 일부를 평가셋으로 사용
eval_samples = valid_sft[:200]

def repetition_rate(text: str, n=2):
    tokens = text.split()
    if len(tokens) < n:
        return 0.0
    ngrams = [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return 1 - (len(set(ngrams)) / len(ngrams))

def evaluate_generation_set(model, samples, mode="single"):
    preds = []
    refs = []

    for row in samples:
        prompt = row["prompt"]
        ref = row["completion"]

        if mode == "single":
            pred = generate_response(model, prompt)
        elif mode == "rm":
            pred, _ = sft_rm_answer(model, prompt, n_candidates=4)
        else:
            raise ValueError("mode should be 'single' or 'rm'")

        preds.append(pred)
        refs.append([ref])

    bleu_score = bleu.compute(predictions=preds, references=refs)
    rouge_score = rouge.compute(predictions=preds, references=[r[0] for r in refs])

    avg_len = np.mean([len(x.split()) for x in preds])
    avg_rep = np.mean([repetition_rate(x) for x in preds])

    return {
        "BLEU": bleu_score["bleu"],
        "ROUGE1": rouge_score["rouge1"],
        "ROUGE2": rouge_score["rouge2"],
        "ROUGEL": rouge_score["rougeL"],
        "AVG_LEN": avg_len,
        "REP_2GRAM": avg_rep,
    }

baseline_metric = evaluate_generation_set(base_model, eval_samples, mode="single")
sft_metric = evaluate_generation_set(sft_model, eval_samples, mode="single")
sft_rm_metric = evaluate_generation_set(sft_model, eval_samples, mode="rm")

metric_df = pd.DataFrame([baseline_metric, sft_metric, sft_rm_metric],
                         index=["baseline", "sft", "sft+rm"])
metric_df

,BLEU,ROUGE1,ROUGE2,ROUGEL,AVG_LEN,REP_2GRAM
baseline,0.000000,0.007720,0.000000,0.007766,3.855,0.0
sft,0.000007,0.029474,0.000625,0.029703,1.000,0.0
sft+rm,0.001755,0.038817,0.001833,0.038982,1.010,0.0


## 16. 디코딩 하이퍼파라미터 탐색
* decoding strategy 비교
* 생성 품질 추가 개선

In [16]:
decode_grid = [
    {"top_p": 0.85, "top_k": 30, "temperature": 0.7},
    {"top_p": 0.90, "top_k": 50, "temperature": 0.8},
    {"top_p": 0.95, "top_k": 50, "temperature": 0.9},
]

def evaluate_decode_setting(model, samples, setting):
    preds = []
    refs = []

    for row in samples[:100]:
        prompt = row["prompt"]
        ref = row["completion"]

        pred = generate_response(
            model,
            prompt,
            top_p=setting["top_p"],
            top_k=setting["top_k"],
            temperature=setting["temperature"],
            max_new_tokens=64
        )
        preds.append(pred)
        refs.append([ref])

    bleu_score = bleu.compute(predictions=preds, references=refs)
    rouge_score = rouge.compute(predictions=preds, references=[r[0] for r in refs])

    return {
        **setting,
        "BLEU": bleu_score["bleu"],
        "ROUGE1": rouge_score["rouge1"],
        "ROUGEL": rouge_score["rougeL"],
        "REP_2GRAM": np.mean([repetition_rate(x) for x in preds])
    }

decode_results = []
for setting in decode_grid:
    decode_results.append(evaluate_decode_setting(sft_model, eval_samples, setting))

decode_df = pd.DataFrame(decode_results).sort_values("ROUGEL", ascending=False)
decode_df

,top_p,top_k,temperature,BLEU,ROUGE1,ROUGEL,REP_2GRAM
0,0.85,30,0.7,0.000000,0.050000,0.050000,0.0
1,0.90,50,0.8,0.000000,0.046667,0.043333,0.0
2,0.95,50,0.9,0.000129,0.029524,0.030000,0.0


## 17. 최종 데모 셀
* 실제 데모 출력
* 최종 개선 체감

In [17]:
demo_prompts = [
    "친구가 자꾸 내 연락을 씹어",
    "면접에서 자기소개 어떻게 해야 해?",
    "잠이 안 와서 너무 힘들어",
    "하루 종일 우울해",
]

for p in demo_prompts:
    print("=" * 100)
    print("[질문]")
    print(p)

    print("\n[Baseline]")
    print(generate_response(base_model, p))

    print("\n[SFT]")
    print(generate_response(sft_model, p))

    best_ans, scored = sft_rm_answer(sft_model, p, n_candidates=4)
    print("\n[SFT + RM]")
    print(best_ans)

    print("\n[RM 후보 점수]")
    for i, (cand, score) in enumerate(scored, 1):
        print(f"{i}. score={score:.4f} | {cand[:120]}")

[질문]
친구가 자꾸 내 연락을 씹어

[Baseline]
)▁등▁여러▁가지▁형태로▁존재한다.
이런▁맥락에서▁볼▁때,▁중국어의▁외래어▁사용이나▁외래어▁학습에▁대해▁어떤▁입장을▁취하는▁것은▁적절한지▁생각해볼▁필요가▁있다.
물론▁외래어▁사용이▁외국어를▁왜곡하거나▁차별하는▁것이▁되어서는▁안▁된다.
외래어는▁그▁사회의▁언어생활에서▁없어서는▁안▁될▁도구이다.
그러므로▁외래어가▁사용되는

[SFT]
'����������.

[SFT + RM]
'��AI�����.<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>

[RM 후보 점수]
1. score=-0.8243 | '��AI�����.<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad
2. score=-2.0876 | '��������,������������������.�������������!<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
3. score=-2.8681 | '����������.����������,��������������������!<pad><pad><pad><pad><pad><pad><pad><pad>
4. score=-6.9419 | '�������.�����������������������,�거���������������
[질문]
면접에서 자기소개 어떻게 해야 해?

[Baseline]
<un